# ML Pipeline DB Template (Local/Azure PostgreSQL)

This notebook uses environment variables so the same code works for local and Azure PostgreSQL.

## 1) Install dependencies (run once)

Uncomment and run the next cell if needed.

In [ ]:
# %pip install pandas sqlalchemy psycopg[binary] python-dotenv scikit-learn

## 2) Configure environment

Create a `.env` file from `.env.example` in this folder.

- For local: `DB_PROFILE=local`, `DB_SSLMODE=prefer`
- For Azure: `DB_PROFILE=azure`, `DB_SSLMODE=require`

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path('.') / '.env'
if env_path.exists():
    load_dotenv(env_path)
else:
    print('No .env found in ML_Pipeline_1. Create one from .env.example.')

profile = os.getenv('DB_PROFILE', 'local').lower()
print(f'Using DB profile: {profile}')

## 3) Build a connection engine

In [ ]:
from sqlalchemy import create_engine, text

def _get_value(key_base: str, profile_name: str) -> str:
    """Prefer profile-specific variables (e.g., AZURE_DB_HOST), fallback to generic (DB_HOST)."""
    profile_key = f"{profile_name.upper()}_{key_base}"
    return os.getenv(profile_key, os.getenv(key_base, ''))

def build_engine(profile_name: str):
    host = _get_value('DB_HOST', profile_name)
    port = _get_value('DB_PORT', profile_name) or '5432'
    name = _get_value('DB_NAME', profile_name)
    user = _get_value('DB_USER', profile_name)
    password = _get_value('DB_PASSWORD', profile_name)
    sslmode = _get_value('DB_SSLMODE', profile_name) or ('require' if profile_name == 'azure' else 'prefer')

    missing = [
        k for k, v in {
            'DB_HOST': host,
            'DB_PORT': port,
            'DB_NAME': name,
            'DB_USER': user,
            'DB_PASSWORD': password,
        }.items() if not v
    ]
    if missing:
        raise ValueError(f'Missing DB config values: {missing}. Check your .env file.')

    # psycopg + SQLAlchemy URL
    url = f'postgresql+psycopg://{user}:{password}@{host}:{port}/{name}?sslmode={sslmode}'
    return create_engine(url, pool_pre_ping=True, future=True)

engine = build_engine(profile)
print('Engine created.')

## 4) Connection test + quick data pull

In [ ]:
import pandas as pd

with engine.connect() as conn:
    print('DB time:', conn.execute(text('select now();')).scalar())

query = """
select
    r.resident_id,
    r.safehouse_id,
    r.case_status,
    r.current_risk_level,
    r.date_of_admission,
    r.date_closed
from residents r
limit 100;
"""

df = pd.read_sql_query(query, engine)
df.head()

## 5) Feature prep starter

In [ ]:
# Example target: whether a case is closed
df['is_closed'] = (df['case_status'] == 'Closed').astype(int)

# Simple date-derived feature example
df['days_since_admission'] = (pd.Timestamp.utcnow().tz_localize(None) - pd.to_datetime(df['date_of_admission'])).dt.days

feature_cols = ['safehouse_id', 'days_since_admission']
X = df[feature_cols].copy()
y = df['is_closed'].copy()

X.head(), y.head()

## 6) Baseline model starter

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() > 1 else None
)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)
preds = model.predict(X_test)

print(classification_report(y_test, preds, zero_division=0))

## Notes

- Keep credentials out of Git.
- Use the same schema/data shape in local and Azure for consistent model behavior.
- Replace the sample query and features with your real pipeline logic.